# Un modelo que escribe, ejecuta y mira

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un agente repite un ciclo: el modelo piensa y escribe código, el programa lo ejecuta y el resultado vuelve al modelo. Si el resultado es difícil de usar, el ciclo se rompe, y el agente solo se detiene cuando se le acaban los pasos. |
| **Producto** | La traza de un agente leída paso a paso, una comparación de dos diseños de herramienta y una regla para elegir entre agente y flujo fijo. |
| **Recursos** | Ollama con `granite4:latest`. La ejecución completa lleva unos cinco minutos, porque los agentes que fallan agotan sus pasos. Gemini es opcional (tres corridas). |

La cooperativa tiene una planilla con las mediciones de ocho parcelas y una regla para marcar las que están en alerta. Alguien propone un agente: que lea la planilla, aplique la regla y conteste. La pregunta práctica no es si el agente puede, sino qué ocurre en cada paso y qué hay que mirar para saber si se le puede confiar la tarea.

**¿Cómo trabaja un agente por dentro, de qué depende que su ciclo funcione y cuándo conviene un flujo fijo en lugar de un agente?**

> **Cómo se trabaja.** En clase se presenta cada tema con el código ya escrito. Después lo explorás por tu cuenta: ejecutás las celdas, leés las notas **Para mirar** y cambiás valores para ver qué pasa. Al final de cada bloque hay un **microlaboratorio**: una tabla te dice qué líneas completar y con qué, y la celda siguiente comprueba si quedó bien.

## Los datos

| Campo | Dato |
|---|---|
| Fuente | `datos/parcelas.csv`: ocho parcelas con temperatura, humedad, lluvia y estado del cultivo. **Datos sintéticos** escritos para el curso a partir de un ejemplo de materiales anteriores; no son mediciones reales de ninguna cooperativa. |
| Unidad de análisis | Una parcela, identificada con una letra de la A a la H |
| Regla de alerta | Una parcela está en alerta si cumple **una o más**: temperatura de 34 °C o más; humedad de 40 % o menos; lluvia igual a 0 mm; estado del cultivo distinto de `normal`. Es una regla didáctica, no un criterio agronómico. |
| Limitaciones | Ocho filas, una regla simple y un solo modelo. Las cifras de aciertos comparan diseños entre sí; no estiman qué tan confiable es un agente en general. |

In [1]:
import utils  # va primero: ordena los avisos de las librerias que siguen
import re
import time

import pandas as pd
from smolagents import CodeAgent, tool

utils.comprobar_modelo_local()
utils.versiones(["smolagents", "ollama", "pandas"])

Ollama responde en http://localhost:11434 y tiene granite4:latest.


{'python': '3.12.13',
 'smolagents': '1.26.0',
 'ollama': '0.6.3',
 'pandas': '3.1.0'}

## 1. El ciclo, paso a paso

Un **agente de código** (`CodeAgent`) trabaja así: recibe una tarea, el modelo **escribe un fragmento de Python**, el programa lo **ejecuta**, lo que se imprimió o devolvió vuelve al modelo como **observación**, y el modelo decide si escribe otro fragmento o entrega la respuesta con `final_answer`. Cada vuelta es un **paso**. `max_steps` es el tope de pasos.

Primero necesitamos una **respuesta de referencia** para saber si el agente acierta. La calculamos con código común, en un par de líneas.

In [2]:
parcelas = utils.cargar_parcelas()


def calcular_alertas(parcelas_a_revisar: list[dict]) -> list[str]:
    """Aplica la regla de alerta a cada parcela.

    Args:
        parcelas_a_revisar: Lista de diccionarios con las mediciones.

    Returns:
        Las letras de las parcelas que cumplen una o mas condiciones.
    """
    en_alerta = []
    for parcela in parcelas_a_revisar:
        if parcela["temperatura_c"] >= 34:
            en_alerta.append(parcela["parcela"])
        elif parcela["humedad_pct"] <= 40:
            en_alerta.append(parcela["parcela"])
        elif parcela["lluvia_mm"] == 0:
            en_alerta.append(parcela["parcela"])
        elif parcela["estado_cultivo"] != "normal":
            en_alerta.append(parcela["parcela"])
    return en_alerta


alertas_correctas = calcular_alertas(parcelas)
print("Parcelas en alerta según la regla:", alertas_correctas)

Parcelas en alerta según la regla: ['B', 'D', 'E', 'F', 'G']


Ahora el agente. Le damos una herramienta que devuelve las parcelas como **lista de diccionarios**, con las claves nombradas en el docstring (el contrato de las herramientas, como vimos en el cuaderno 01).

In [3]:
@tool
def leer_parcelas() -> list:
    """Devuelve las mediciones de las parcelas como una lista de diccionarios.

    Cada diccionario tiene exactamente estas claves: "parcela" (letra),
    "temperatura_c" (numero), "humedad_pct" (numero), "lluvia_mm" (numero)
    y "estado_cultivo" (texto).

    Returns:
        Lista con un diccionario por parcela.
    """
    return utils.cargar_parcelas()


modelo = utils.crear_modelo_local()

reglas = (
    "Una parcela está en alerta si cumple una o más de estas condiciones: "
    "temperatura_c >= 34; humedad_pct <= 40; lluvia_mm == 0; "
    "estado_cultivo distinto de 'normal'."
)
tarea = (
    f"Usá la herramienta leer_parcelas. {reglas} Devolvé con final_answer "
    "el texto con las letras de las parcelas en alerta, separadas por comas."
)

agente = CodeAgent(
    tools=[leer_parcelas], model=modelo, max_steps=5, verbosity_level=-1
)
inicio = time.perf_counter()
respuesta = agente.run(tarea)
segundos = time.perf_counter() - inicio
print("Respuesta del agente:", respuesta)
print(f"Tardó {segundos:.0f} segundos")

Respuesta del agente: B,D,E,F,G
Tardó 5 segundos


> **Para mirar.**
>
> - El agente resolvió la tarea en un paso y la respuesta coincide con la regla: `B,D,E,F,G`.
> - Ese paso consistió en que el modelo escribió un fragmento de Python que llamó a `leer_parcelas()`, aplicó la regla a cada parcela y entregó el resultado con `final_answer`.
> - Que coincida con la regla se sabe porque la calculamos aparte; sin esa referencia, la respuesta parecería correcta de todas formas.

### Microlaboratorio 1 — Leer la traza

**Tarea.** Escribir la función que resume qué hizo el agente en cada paso.  
**Producto.** `resumir_pasos`, comprobada con el agente de la sección 1.  
**Antes de empezar.** Sección 1.

El agente guarda su historia en `agente.memory.steps`: una lista con la tarea y un `ActionStep` por cada paso. Cada `ActionStep` tiene `step_number`, `model_output` (lo que escribió el modelo), `observations` (lo que volvió de ejecutarlo) y `error` (`None` si no hubo error).

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) condición del `if` | verdadera solo para los pasos de acción, no para la tarea | `type(paso).__name__ == "ActionStep"` |
| (2) `"paso"` | el número del paso | `paso.step_number` |
| (3) `"error"` | `True` si el paso tuvo un error | comparar `paso.error` con `None` usando `is not` |

**Comprobación.** Hay una fila por paso de acción, la primera es el paso 1 y la columna `error` es booleana.

> **Ayuda 1.** `agente.memory.steps` mezcla tipos de paso; la condición los separa por el nombre de su clase.  
> **Ayuda 2.** Un paso sin error tiene `paso.error` igual a `None`.

In [4]:
def resumir_pasos(agente_a_revisar: CodeAgent) -> list[dict]:
    """Resume los pasos de accion que dio un agente.

    Args:
        agente_a_revisar: Agente de smolagents que ya ejecuto una tarea.

    Returns:
        Una fila por paso, con el numero, lo que escribio el modelo, lo
        que volvio de ejecutarlo y si hubo un error.
    """
    filas = []
    for paso in agente_a_revisar.memory.steps:
        if False:  # COMPLETAR (1)
            escrito = " ".join((paso.model_output or "").split())
            devuelto = " ".join((paso.observations or "").split())
            filas.append({
                "paso": 0,  # COMPLETAR (2)
                "escribio el modelo": escrito[:70],
                "volvio": devuelto[:70],
                "error": False,  # COMPLETAR (3)
            })
    return filas


pasos_resumidos = resumir_pasos(agente)
cantidad_de_pasos = 0
for paso_de_memoria in agente.memory.steps:
    if type(paso_de_memoria).__name__ == "ActionStep":
        cantidad_de_pasos += 1

utils.comprobar(
    len(pasos_resumidos) == cantidad_de_pasos and cantidad_de_pasos > 0,
    "revisa (1): tiene que haber una fila por cada ActionStep",
)
utils.comprobar(
    len(pasos_resumidos) > 0 and pasos_resumidos[0]["paso"] == 1,
    "revisa (2): el primer paso tiene que ser el numero 1",
)

Todavia no: revisa (1): tiene que haber una fila por cada ActionStep
Todavia no: revisa (2): el primer paso tiene que ser el numero 1


A partir de acá usamos `utils.resumir_pasos`, que hace lo mismo que tu función, para que las tablas siguientes sean legibles aunque todavía no la hayas resuelto.

In [5]:
utils.tabla(utils.resumir_pasos(agente))

,paso,escribio el modelo,volvio,error
0,1,<code> parcelas = leer_parcelas() alertas = [] for parcela in parcelas,"Execution logs: Last output from code snippet: B,D,E,F,G",False


> **Para mirar.**
>
> - Hay una sola fila: en un paso el modelo escribió el código completo y lo que volvió de ejecutarlo fue `B,D,E,F,G`, sin error.
> - La traza permite ver *qué código se ejecutó* y *qué devolvió*. Cuando algo sale mal, es lo primero que se lee.

## 2. De qué depende que el ciclo funcione

La herramienta de la sección 1 devuelve datos **ya estructurados**. ¿Qué pasa si devuelve lo mismo pero como **texto CSV**, que es como suele llegar una planilla? Armamos una segunda herramienta y corremos cada diseño tres veces. Un agente nuevo en cada corrida, el mismo modelo, la misma tarea.

In [6]:
@tool
def leer_parcelas_en_texto() -> str:
    """Devuelve las mediciones de las parcelas en formato CSV.

    Returns:
        Texto CSV con las mediciones de las parcelas.
    """
    return (utils.RUTA_DATOS / "parcelas.csv").read_text(encoding="utf-8")


def letras_de(respuesta: object) -> set:
    """Extrae las letras de parcela de una respuesta corta.

    Args:
        respuesta: Lo que devolvio el agente.

    Returns:
        Conjunto de letras de la A a la H. Si la respuesta es larga o no
        contiene letras sueltas, el conjunto queda vacio.
    """
    texto = str(respuesta)
    if len(texto) > 40:
        return set()
    return set(re.findall(r"\b[A-H]\b", texto))


def correr_agentes(
    herramienta: object, nombre_de_herramienta: str, cantidad: int
) -> tuple:
    """Corre varios agentes nuevos con la misma tarea y mide que pasa.

    Args:
        herramienta: Herramienta que se le da al agente.
        nombre_de_herramienta: Nombre con el que la nombra la tarea.
        cantidad: Cuantas corridas hacer.

    Returns:
        Una tupla: las filas (una por corrida, con pasos, segundos y si
        acerto) y el ultimo agente, para poder mirar su traza.
    """
    tarea_de_la_herramienta = (
        f"Usá la herramienta {nombre_de_herramienta}. {reglas} Devolvé con "
        "final_answer el texto con las letras de las parcelas en alerta, "
        "separadas por comas."
    )
    filas = []
    for corrida in range(1, cantidad + 1):
        agente_nuevo = CodeAgent(
            tools=[herramienta], model=modelo, max_steps=5, verbosity_level=-1
        )
        inicio = time.perf_counter()
        respuesta_nueva = agente_nuevo.run(tarea_de_la_herramienta)
        segundos_de_corrida = time.perf_counter() - inicio
        pasos_dados = len(utils.resumir_pasos(agente_nuevo))
        filas.append({
            "corrida": corrida,
            "pasos": pasos_dados,
            "segundos": round(segundos_de_corrida),
            "acertó": letras_de(respuesta_nueva) == set(alertas_correctas),
        })
    return filas, agente_nuevo

In [7]:
filas_estructuradas, _ = correr_agentes(leer_parcelas, "leer_parcelas", 3)
filas_en_texto, ultimo_agente_en_texto = correr_agentes(
    leer_parcelas_en_texto, "leer_parcelas_en_texto", 3
)

diseno_estructurado = utils.tabla(filas_estructuradas)
diseno_estructurado.insert(0, "herramienta", "lista de diccionarios")
diseno_en_texto = utils.tabla(filas_en_texto)
diseno_en_texto.insert(0, "herramienta", "texto CSV")
pd.concat([diseno_estructurado, diseno_en_texto], ignore_index=True)

,herramienta,corrida,pasos,segundos,acertó
0,lista de diccionarios,1,1,5,True
1,lista de diccionarios,2,1,5,True
2,lista de diccionarios,3,1,5,True
3,texto CSV,1,6,62,False
4,texto CSV,2,6,68,False
5,texto CSV,3,6,70,False


> **Para mirar.**
>
> - Con la lista de diccionarios, las tres corridas acertaron en un paso y en pocos segundos.
> - Con texto CSV, las tres corridas agotaron el tope de pasos (la columna `pasos` llega a 6 con un tope de 5), tardaron decenas de segundos y ninguna acertó.
> - El modelo, la tarea y los datos eran los mismos: cambió cómo la herramienta entrega los datos.

In [8]:
# Primer paso con error del agente que recibio el texto CSV.
paso_con_error = None
cantidad_con_error = 0
for paso in ultimo_agente_en_texto.memory.steps:
    if type(paso).__name__ == "ActionStep" and paso.error is not None:
        cantidad_con_error = cantidad_con_error + 1
        if paso_con_error is None:
            paso_con_error = paso

print(f"Pasos con error en la ultima corrida con texto: {cantidad_con_error}")
lineas_del_error = str(paso_con_error.error).splitlines()
print(lineas_del_error[-1])

Pasos con error en la ultima corrida con texto: 5
        alertas.append(parcela)' due to: InterpreterError: Cannot unpack tuple of wrong size


> **Para mirar.**
>
> - El modelo trató el texto como si ya fuera una lista de filas: recorre `parcelas_en_texto` y llama a `split(',')` sobre cada elemento, pero iterar un texto da caracteres, no renglones. La última línea del error, impresa arriba, muestra dónde se rompió.
> - El agente vio el error en cada paso y probó variantes del mismo enfoque. No corrigió la causa, que era separar primero el texto en renglones.
> - Un modelo más grande, o una instrucción que explique cómo viene el texto, podrían salvarlo. Acá medimos la diferencia de diseño con este modelo.

## 3. Cuando se acaban los pasos

`max_steps` evita que un agente que no avanza siga para siempre. Pero ¿qué devuelve cuando se queda sin pasos? Corremos el diseño en texto con solo dos pasos permitidos.

In [9]:
agente_corto = CodeAgent(
    tools=[leer_parcelas_en_texto], model=modelo, max_steps=2,
    verbosity_level=-1,
)
respuesta_corta = agente_corto.run(
    f"Usá la herramienta leer_parcelas_en_texto. {reglas} Devolvé con "
    "final_answer el texto con las letras de las parcelas en alerta, "
    "separadas por comas."
)
print("Lo que devolvió el agente:")
print(str(respuesta_corta)[:300])
print()
print("Letras que se pudieron extraer:", letras_de(respuesta_corta))

Lo que devolvió el agente:
<code>
parcelas_en_texto = leer_parcelas_en_texto()
alertas = []
for parcela in parcelas_en_texto:
    temperatura_c, humedad_pct, lluvia_mm, estado_cultivo = parcela.split(',')
    if (temperatura_c >= 34) or (humedad_pct <= 40) or (lluvia_mm == 0) or (estado_cultivo != 'normal'):
        alertas.a

Letras que se pudieron extraer: set()


> **Para mirar.**
>
> - Con dos pasos el agente no llegó a una respuesta: lo que devolvió es un fragmento de código a medio escribir, no una lista de letras.
> - `letras_de` extrajo un conjunto vacío: validar el formato de la salida permite detectar que no es una respuesta.
> - Un programa que hubiera tomado ese texto como respuesta habría guardado código como si fuera un informe.

## 4. Qué puede ejecutar el agente

Un `CodeAgent` ejecuta el código que escribe el modelo **en tu computadora**. Por defecto lo hace con un intérprete restringido: solo permite importar algunos módulos. Probamos ese filtro sin usar el modelo: le pasamos código directo al ejecutor.

In [10]:
agente_de_prueba = CodeAgent(tools=[], model=modelo, verbosity_level=-1)

# error intencional: InterpreterError (el modulo os no esta autorizado)
try:
    agente_de_prueba.python_executor("import os\nos.listdir('.')")
except Exception as error:
    print(type(error).__name__)
    print(str(error)[:260])

InterpreterError
Code execution failed at line 'import os' due to: InterpreterError: Import of os is not allowed. Authorized imports are: ['random', 'itertools', 'unicodedata', 'statistics', 'datetime', 'stat', 'queue', 'collections', 'time', 're', 'math']


> **Para mirar.**
>
> - El intérprete rechazó `import os` y mostró la lista de módulos permitidos (entre ellos `datetime`, `math`, `re` y `time`).
> - Es un filtro útil contra errores y usos accidentales, no una barrera de seguridad: ejecuta código en tu computadora. Para código de origen no confiable hace falta un entorno aislado.

## 5. Preguntas distintas, mismo agente

Un flujo fijo (como `calcular_alertas`) responde una sola pregunta. Un agente puede responder preguntas que nadie anticipó. Probamos tres, con respuestas que calculamos aparte.

In [11]:
preguntas_nuevas = [
    {
        "pregunta": (
            "¿Cuál es la parcela con mayor temperatura "
            "y cuánta lluvia tuvo?"
        ),
        "debe_incluir": ["D", "0"],
    },
    {
        "pregunta": "¿Cuántas parcelas tienen una humedad menor a 40?",
        "debe_incluir": ["3"],
    },
    {
        "pregunta": (
            "¿Cuál es la lluvia promedio de las parcelas, "
            "en milímetros?"
        ),
        "debe_incluir": ["6.25"],
    },
]

filas_de_preguntas = []
for caso in preguntas_nuevas:
    agente_de_pregunta = CodeAgent(
        tools=[leer_parcelas], model=modelo, max_steps=5, verbosity_level=-1
    )
    inicio = time.perf_counter()
    respuesta_de_pregunta = agente_de_pregunta.run(
        f"Usá la herramienta leer_parcelas. {caso['pregunta']} "
        "Devolvé con final_answer una frase con los valores concretos."
    )
    texto = str(respuesta_de_pregunta).replace(",", ".")
    correcta = True
    for dato in caso["debe_incluir"]:
        if dato not in texto:
            correcta = False
    filas_de_preguntas.append({
        "pregunta": caso["pregunta"],
        "respuesta": str(respuesta_de_pregunta)[:90],
        "pasos": len(utils.resumir_pasos(agente_de_pregunta)),
        "segundos": round(time.perf_counter() - inicio),
        "correcta": correcta,
    })

utils.tabla(filas_de_preguntas)

,pregunta,respuesta,pasos,segundos,correcta
0,¿Cuál es la parcela con mayor temperatura y cuánta lluvia tuvo?,La parcela con mayor temperatura es 36.0°C y recibió 22.0 mm de lluvia.,2,18,False
1,¿Cuántas parcelas tienen una humedad menor a 40?,There are 3 parcels with humidity below 40.,1,4,True
2,"¿Cuál es la lluvia promedio de las parcelas, en milímetros?",El promedio de lluvia en las parcelas es de 6.25 mm.,1,5,True


> **Para mirar.**
>
> - Dos de las tres respuestas son correctas.
> - La incorrecta es plausible: dice que la parcela con mayor temperatura recibió 22 mm. La temperatura de 36 °C es la de la parcela D, pero los 22 mm son de la parcela C. El agente mezcló valores de dos parcelas y entregó una frase prolija.
> - La respuesta sobre la humedad llegó en inglés aunque la pregunta estaba en español; el dato (3 parcelas) es correcto.
> - Sin la respuesta calculada aparte no habríamos detectado el primer error.

## 6. El mismo agente con un modelo remoto

Repetimos la prueba que más costó: el diseño con **texto CSV**, ahora con Gemini como modelo del agente. Es opcional y hace tres corridas, con una pausa entre una y otra por los límites del plan gratuito.

In [12]:
if utils.hay_clave("GEMINI_API_KEY"):
    modelo_remoto = utils.crear_modelo_gemini()
    filas_remotas = []
    for corrida in range(1, 4):
        agente_remoto = CodeAgent(
            tools=[leer_parcelas_en_texto], model=modelo_remoto,
            max_steps=5, verbosity_level=-1,
        )
        try:
            respuesta_remota = agente_remoto.run(
                f"Usá la herramienta leer_parcelas_en_texto. {reglas} "
                "Devolvé con final_answer el texto con las letras de las "
                "parcelas en alerta, separadas por comas."
            )
        except Exception as error:
            respuesta_remota = f"Error: {error}"
        filas_remotas.append({
            "corrida": corrida,
            "pasos": len(utils.resumir_pasos(agente_remoto)),
            "acertó": letras_de(respuesta_remota) == set(alertas_correctas),
        })
        time.sleep(6)
    tabla_remota = utils.tabla(filas_remotas)
else:
    tabla_remota = None
    utils.avisar_sin_clave("GEMINI_API_KEY")

tabla_remota

Sin GEMINI_API_KEY: se omite esta parte. Copia .env.example como .env, completa la clave y reinicia el kernel.


> **Para mirar.** Con clave, compará con la tabla de la sección 2. Si Gemini resuelve el diseño en texto que el modelo local no pudo, la diferencia está en el modelo, no en el ciclo; si falla igual, el problema es de la herramienta. Sin clave, esta parte queda sin ejecutar.

### Microlaboratorio 2 — ¿Agente o flujo fijo?

**Tarea.** Elegir, para cuatro tareas, entre un flujo fijo y un agente, según lo que mediste.  
**Producto.** `eleccion` y `mi_decision`.  
**Antes de empezar.** Secciones 2, 3 y 5.

Las opciones son `"flujo fijo"` y `"agente"`.

| Tarea | Qué tiene que quedar |
|---|---|
| (1) calcular todas las mañanas las parcelas en alerta, siempre con la misma regla | una opción, según la sección 1 |
| (2) contestar preguntas distintas de distintas personas sobre la planilla | una opción, según la sección 5 |
| (3) enviar el aviso de alerta por correo a los productores | una opción, pensando en qué pasa si el paso falla |
| (4) mirar una planilla nueva que nadie conoce para ver qué patrones tiene | una opción, según las secciones 2 y 5 |

En `mi_decision`, escribí al menos 40 palabras: **qué evidencia de este cuaderno te hizo elegir un flujo fijo para alguna tarea, y qué control le agregarías al agente en otra**.

> **Ayuda 1.** En la sección 2, el mismo modelo pasó de acertar siempre a no acertar nunca por un detalle de la herramienta.  
> **Ayuda 2.** Un correo enviado no se puede deshacer; una respuesta equivocada en pantalla, sí.

In [13]:
eleccion = {
    "alertas de todas las mananas": "agente",  # COMPLETAR (1)
    "preguntas distintas sobre la planilla": "agente",  # COMPLETAR (2)
    "aviso por correo": "agente",  # COMPLETAR (3)
    "planilla nueva": "agente",  # COMPLETAR (4)
}

opciones_validas = ["flujo fijo", "agente"]
todas_validas = True
for tarea_elegida, opcion in eleccion.items():
    if opcion not in opciones_validas:
        todas_validas = False
utils.comprobar(todas_validas, "usa solo flujo fijo o agente")

esperadas = {
    "alertas de todas las mananas": "flujo fijo",
    "preguntas distintas sobre la planilla": "agente",
    "aviso por correo": "flujo fijo",
    "planilla nueva": "agente",
}
a_revisar = []
for tarea_elegida, opcion in eleccion.items():
    if opcion != esperadas[tarea_elegida]:
        a_revisar.append(tarea_elegida)
utils.comprobar(
    len(a_revisar) == 0,
    "revisa estas tareas: " + ", ".join(a_revisar),
)

mi_decision = ""
utils.comprobar(
    len(mi_decision.split()) >= 40,
    "escribi al menos 40 palabras en mi_decision",
)

Correcto.
Todavia no: revisa estas tareas: alertas de todas las mananas, aviso por correo
Todavia no: escribi al menos 40 palabras en mi_decision


## Qué te llevás

- Un agente de código repite **escribir, ejecutar y observar**. La traza (`memory.steps`) permite ver cada paso y es lo primero que se mira cuando algo sale mal.
- El ciclo depende de la **forma en que la herramienta entrega los datos**: el mismo modelo y la misma tarea pasaron de resolverse en un paso a agotar todos los pasos solo por devolver texto CSV en lugar de una lista de diccionarios.
- Un agente que se queda sin pasos **no avisa con claridad**: devuelve algo que parece una respuesta. Hay que validar el formato de lo que devuelve y decidir **cuándo conviene un flujo fijo**: la misma tarea, siempre igual, sin efectos que no se deshacen.

**Sigue:** cómo se le dan criterios explícitos a un agente, cómo se verifica su trabajo contra una regla y dónde entra una persona que aprueba (cuaderno 03).